# Notebook 06 — Saved-prediction audit and participant uncertainty

Use a **CPU runtime** and run the single code cell. Keep the original `SensorRepair_ICDSIS` Drive project and all completed Notebook 05 outputs. No dataset download, GPU, model fitting, inference, or threshold changes are needed.

The notebook checks prior-file hashes and reproduces the 15 configuration results directly from their saved predictions. It verifies the condition, fold, seed, participant, and aggregate tables before reporting anything new. It checks disjoint DSADS test folds and matching window identities across seeds. The labeled oracle stays diagnostic.

**Two different metrics are reported explicitly.**

1. **Primary:** the unchanged Notebook 05 Macro-F1: compute per condition and test fold, average conditions within a scope, average four DSADS folds within a seed, then average the three seeds. Three-seed SD remains descriptive. No confidence interval is attached to this primary metric.
2. **Secondary:** first compute Macro-F1 separately for each participant, over that participant's present classes. Average the condition scores and three saved seed scores for that participant, then average participants equally. A paired percentile bootstrap resamples the **8 DSADS or 9 HAR participants** 10,000 times, keeping each participant's seeds and fault views together and using the same sampled participant indices for all methods and scopes. Its 95% intervals refer **only to this secondary metric**. A participant-deletion table shows sensitivity without refitting any model.

These are exploratory intervals conditional on the saved fitted models. They exclude retraining variability, do not resolve dependence caused by overlapping cross-validation training data, and are not simultaneous intervals across comparisons. Small participant counts limit interpretation. No p-values, statistical-significance claims, noninferiority claims, certified harm bound, or population guarantees are produced. References: [paired bootstrap documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) and [Bengio and Grandvalet, JMLR 2004](https://jmlr.org/papers/v5/grandvalet04a.html).

All ten non-oracle comparators are retained in the comparison CSV. The printed and plotted views show keep, residual masking, and confidence selection. Comparisons after inspecting the study are identified as exploratory; they never alter the saved method. The harm audit reports per-run rates, including violations hidden by a low overall mean, and separates rates per window from rates conditional on intervention or original correctness.

Outputs include comparison and harm tables, participant scores and deletion sensitivity, per-condition diagnostics, saved bootstrap indices/samples, and publication-resolution PDF/PNG figures. Error bars in the primary-results figure are **seed SD**, while the participant-interval figure explicitly uses the **secondary metric**. Low intervention rates do not establish compute savings because the original implementation evaluated every candidate action.

When complete, share the printed output and the three named result files. Earlier results remain unchanged. This notebook does not establish novelty or guarantee a publishable improvement.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib
import json
import os
import shutil
import time
import zlib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

PROJECT=Path('/content/drive/MyDrive/SensorRepair_ICDSIS')
SOURCE=PROJECT/'outputs'/'notebook05_v1'
OUTPUT=PROJECT/'outputs'/'notebook06_v1'
LOCAL=Path('/content/sensorrepair_notebook06_v1')
CODE_VERSION='sensorrepair_nb06_v1'
CELL_SIGNATURE='f11927e9838850120d451166d4e39632b5f83d620fb3d0ef8d439c143fb3508c'
BOOTSTRAP_RESAMPLES=10000
BOOTSTRAP_SEED=20261021
CONFIDENCE_LEVEL=.95
FAMILIES=['harm_aware_policy','benefit_only_policy','confidence_margin_tuned','residual_mask_tuned','residual_reconstruct_tuned']
BASELINES=['keep','mask_suspect','reconstruct_suspect','residual_gate_mask','residual_gate_reconstruct','confidence_select','oracle_diagnostic']
METHODS=BASELINES+FAMILIES
COMPARATORS=[m for m in METHODS if m not in ['harm_aware_policy','oracle_diagnostic']]
DISPLAY_COMPARATORS=['keep','residual_gate_mask','confidence_select']
CONDITIONS=[('clean',None,None)]+[(f'{f}_{i+1}',f,i) for f in ['dropout','bias','gain','stuck'] for i in range(3)]
SCOPES={'clean':['clean'],'known_faults':[c for c,f,_ in CONDITIONS if f in ['dropout','bias','gain']],
        'unseen_stuck':[c for c,f,_ in CONDITIONS if f=='stuck']}
NUMERIC=['macro_f1','accuracy','intervention_rate','helpful_rate','harmful_rate']
OUTPUT.mkdir(parents=True,exist_ok=True)
LOCAL.mkdir(parents=True,exist_ok=True)
print('NOTEBOOK 06 | saved-prediction audit and participant uncertainty | CPU | one executable cell',flush=True)
print('No training, new test predictions, threshold changes, or GPU required.',flush=True)



def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def json_signature(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def save_csv(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    value.to_csv(temporary, index=False)
    os.replace(temporary, path)


def save_npz(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + '.partial')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(temporary, path)


def macro_f1(y, prediction, classes):
    matrix = np.bincount(y * classes + prediction, minlength=classes * classes).reshape(classes, classes)
    numerator = 2.0 * matrix.diagonal()
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(numerator, denominator, out=np.zeros(classes), where=denominator > 0).mean())


def evaluate_prediction_arrays(arrays,classes):
    methods=arrays['methods'].tolist()
    conditions=arrays['conditions'].tolist()
    y=arrays['y']
    predictions,actions=arrays['predictions'],arrays['choices']
    assert methods==METHODS and conditions==[c[0] for c in CONDITIONS]
    assert predictions.shape==actions.shape==(len(methods),len(conditions),len(y))
    assert np.issubdtype(predictions.dtype,np.integer) and predictions.min()>=0 and predictions.max()<classes
    assert np.issubdtype(actions.dtype,np.integer) and actions.min()>=0
    original_correct=predictions[methods.index('keep')]==y[None,:]
    rows,summary,participants=[],[],[]
    for m,method in enumerate(methods):
        correct=predictions[m]==y[None,:]
        method_rows=[]
        for d,condition in enumerate(conditions):
            helpful=float((~original_correct[d]&correct[d]).mean())
            harmful=float((original_correct[d]&~correct[d]).mean())
            accuracy=float(correct[d].mean())
            assert abs(accuracy-original_correct[d].mean()-helpful+harmful)<1e-10
            method_rows.append({'method':method,'condition':condition,'windows':len(y),
                'macro_f1':macro_f1(y,predictions[m,d],classes),'accuracy':accuracy,
                'intervention_rate':float((actions[m,d]!=0).mean()),'helpful_rate':helpful,'harmful_rate':harmful})
        rows.extend(method_rows)
        for scope,names in SCOPES.items():
            part=[row for row in method_rows if row['condition'] in names]
            summary.append({'method':method,'scope':scope,'unique_test_windows':len(y),
                **{key:float(np.mean([row[key] for row in part])) for key in
                  ['macro_f1','accuracy','intervention_rate','helpful_rate','harmful_rate']}})
        if method=='oracle_diagnostic':
            continue
        for subject in np.unique(arrays['subjects']):
            use=arrays['subjects']==subject
            present=np.unique(y[use])
            for scope,names in SCOPES.items():
                f1s,accs,helps,harms,interventions=[],[],[],[],[]
                for condition in names:
                    d=conditions.index(condition)
                    cm=np.bincount(y[use]*classes+predictions[m,d,use],minlength=classes*classes).reshape(classes,classes)
                    denom=cm.sum(0)+cm.sum(1)
                    values=np.divide(2.*cm.diagonal(),denom,out=np.zeros(classes),where=denom>0)
                    f1s.append(float(values[present].mean()));accs.append(float(correct[d,use].mean()))
                    helps.append(float((~original_correct[d,use]&correct[d,use]).mean()))
                    harms.append(float((original_correct[d,use]&~correct[d,use]).mean()))
                    interventions.append(float((actions[m,d,use]!=0).mean()))
                participants.append({'subject':int(subject),'method':method,'scope':scope,'windows':int(use.sum()),
                    'classes_present':len(present),'macro_f1_present_classes':float(np.mean(f1s)),
                    'accuracy':float(np.mean(accs)),'helpful_rate':float(np.mean(helps)),
                    'harmful_rate':float(np.mean(harms)),'intervention_rate':float(np.mean(interventions))})
    return rows,summary,participants




inventory_path=SOURCE/'notebook05_artifact_inventory.json'
if not inventory_path.is_file():
    raise FileNotFoundError(f'Missing {inventory_path}. Finish Notebook 05 first.')
records=json.loads(inventory_path.read_text())
inventory={r['filename']:r for r in records}
assert len(records)==len(inventory)


def verified_copy(path,relative,expected):
    target=LOCAL/'inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_file() and file_sha256(target)==expected:
        return target
    partial=target.with_name(target.name+'.partial')
    shutil.copyfile(path,partial)
    assert file_sha256(partial)==expected,f'Input hash mismatch: {relative}'
    os.replace(partial,target)
    return target


def root_file(relative):
    return verified_copy(SOURCE/relative,'nb05/'+relative,inventory[relative]['sha256'])


report=json.loads(root_file('notebook05_run_report.json').read_text())
manifest=json.loads(root_file('notebook05_study_manifest.json').read_text())
assert report['study_complete'] and report['configurations_complete']==report['configurations_required']==15
assert report['pending_configurations']==[] and report['test_based_threshold_changes']==0
assert report['study_signature']==manifest['signature']
jobs=manifest['plan']['jobs']
results=report['completed_results']
identity=lambda x:(x['dataset'],x['fold'],int(x['seed']))
assert len({identity(r) for r in results})==15 and {identity(r) for r in results}=={identity(j) for j in jobs}
SEEDS=sorted({int(j['seed']) for j in jobs})
assert SEEDS==[2026,2027,2028]
PRIMARY=report['primary_comparators']
protocol_path=verified_copy(PROJECT/'processed'/'v1'/'protocol.json','prepared/protocol.json',
                            manifest['plan']['protocol_sha256'])
protocol=json.loads(protocol_path.read_text())
CLASSES={ds:len(protocol['datasets'][ds]['class_names']) for ds in ['dsads','har']}
assert CLASSES=={'dsads':19,'har':6}
plan={'code_version':CODE_VERSION,'cell_signature':CELL_SIGNATURE,'source_study_signature':report['study_signature'],
      'source_inventory_sha256':file_sha256(inventory_path),
      'source_report_sha256':inventory['notebook05_run_report.json']['sha256'],
      'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,'confidence_level':CONFIDENCE_LEVEL,
      'comparators':COMPARATORS,'primary_comparators':PRIMARY,
      'primary_metric':'exact Notebook 05 condition/fold/seed averaged Macro-F1; point estimates and seed SD only',
      'secondary_metric':'equal-participant mean of within-participant Macro-F1 on present classes, first averaged over conditions and the three saved seeds',
      'bootstrap':'paired participant percentile bootstrap; same sampled participants for every method and scope; all saved seed and fault views of a participant stay together',
      'bootstrap_limitations':'exploratory post-analysis; 8 DSADS and 9 HAR participants; conditional on saved fitted models; excludes retraining variability and does not resolve dependence from overlapping CV training sets; intervals are not simultaneous',
      'primary_ci_from_secondary_bootstrap':False,'p_values_or_significance_claims':False,
      'participant_deletion':'remove one participant score from the secondary mean; no model refitting',
      'new_models_or_policy_settings':False,
      'references':['https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html',
                    'https://jmlr.org/papers/v5/grandvalet04a.html']}
ANALYSIS_SIGNATURE=json_signature(plan)
analysis_manifest=OUTPUT/'notebook06_analysis_manifest.json'
if analysis_manifest.exists():
    assert json.loads(analysis_manifest.read_text())['signature']==ANALYSIS_SIGNATURE, (
        'Analysis inputs or definitions changed. Restore the original configuration instead of overwriting its results.')
else:
    save_json(analysis_manifest,{'signature':ANALYSIS_SIGNATURE,'plan':plan})
print('Analysis plan recorded:',ANALYSIS_SIGNATURE[:16],flush=True)


def assert_table_parity(computed,original,keys,columns):
    assert not computed.duplicated(keys).any() and not original.duplicated(keys).any()
    a=computed.set_index(keys).sort_index()
    b=original.set_index(keys).sort_index()
    assert a.index.equals(b.index),'Table row identities changed.'
    for col in columns:
        assert np.allclose(a[col].to_numpy(dtype=float),b[col].to_numpy(dtype=float),atol=1e-12,rtol=0,equal_nan=True),f'Metric parity failed: {col}'


def run_file(result,filename):
    mapping={r['filename']:r for r in result['artifacts']}
    relative=f'runs/{result["dataset"]}_{result["fold"]}_seed_{result["seed"]}/{filename}'
    return verified_copy(SOURCE/relative,'nb05/'+relative,mapping[filename]['sha256'])


def load_and_reproduce():
    rows,summary,participants,harm_rows,fold_policy_rows=[],[],[],[],[]
    metadata={}
    for result in results:
        job={key:result[key] for key in ['dataset','fold','seed']}
        dataset=job['dataset']
        selected=json.loads(run_file(result,'selected_policy.json').read_text())
        main_settings=selected['selected_methods']['harm_aware_policy']['settings']
        assert main_settings==result['selected_main_policy']
        with np.load(run_file(result,'test_predictions.npz'),allow_pickle=False) as saved:
            assert saved['signature'].item()==result['signature']
            arrays={k:saved[k] for k in ['methods','conditions','predictions','choices','y','subjects','window_ids','source_indices']}
        N=len(arrays['y'])
        assert N==result['unique_test_windows'] and len(np.unique(arrays['window_ids']))==N
        assert arrays['y'].ndim==arrays['subjects'].ndim==arrays['window_ids'].ndim==arrays['source_indices'].ndim==1
        assert all(len(arrays[k])==N for k in ['subjects','window_ids','source_indices'])
        assert np.array_equal(np.unique(arrays['y']),np.arange(CLASSES[dataset]))
        assert np.unique(arrays['subjects']).tolist()==result['test_participants']
        expected_actions=1+2*len(protocol['datasets'][dataset]['sensor_groups'])
        assert arrays['choices'].max()<expected_actions
        assert np.all(arrays['choices'][METHODS.index('keep')]==0)
        keep=arrays['predictions'][METHODS.index('keep')]
        main=arrays['predictions'][METHODS.index('harm_aware_policy')]
        actions=arrays['choices'][METHODS.index('harm_aware_policy')]
        structural_keep=main_settings['threshold']>=1 and main_settings['harm_penalty']>=0
        assert structural_keep==result['structural_keep_only']
        if structural_keep:
            assert np.all(actions==0) and np.array_equal(main,keep)
        r,s,p=evaluate_prediction_arrays(arrays,CLASSES[dataset])
        run_frames={name:pd.DataFrame([{**job,**row} for row in values]) for name,values in
                    [('per_condition',r),('test_summary',s),('per_participant',p)]}
        for filename,keys,cols in [('per_condition',['dataset','fold','seed','method','condition'],NUMERIC),
                                   ('test_summary',['dataset','fold','seed','method','scope'],NUMERIC),
                                   ('per_participant',['dataset','fold','seed','subject','method','scope'],
                                    ['macro_f1_present_classes','accuracy','intervention_rate','helpful_rate','harmful_rate'])]:
            assert_table_parity(run_frames[filename],pd.read_csv(run_file(result,filename+'.csv')),keys,cols)
        rows.extend(run_frames['per_condition'].to_dict('records'))
        summary.extend(run_frames['test_summary'].to_dict('records'))
        participants.extend(run_frames['per_participant'].to_dict('records'))
        meta=pd.DataFrame({key:arrays[key] for key in ['window_ids','source_indices','y','subjects']}).sort_values('window_ids').reset_index(drop=True)
        metadata[identity(job)]=meta
        original_correct=keep==arrays['y'][None,:]
        selected_correct=main==arrays['y'][None,:]
        for scope,names in SCOPES.items():
            ds=[arrays['conditions'].tolist().index(c) for c in names]
            correct=selected_correct[ds];original=original_correct[ds];chosen=actions[ds]
            harmful=original&~correct;helpful=~original&correct;interventions=chosen!=0
            H,F,I,O=int(harmful.sum()),int(helpful.sum()),int(interventions.sum()),int(original.sum())
            row={**job,'scope':scope,'unique_windows':N,'condition_views':len(ds),'window_condition_cases':N*len(ds),
                 'harmful_cases':H,'helpful_cases':F,'intervention_cases':I,'keep_correct_cases':O,
                 'harm_all_windows':float(harmful.mean()),'help_all_windows':float(helpful.mean()),
                 'intervention_rate':float(interventions.mean()),
                 'harm_given_keep_correct':H/O if O else None,'harm_given_intervention':H/I if I else None,
                 'prediction_change_rate':float((main[ds]!=keep[ds]).mean()),
                 'empirical_harm_le_1pct':bool(harmful.mean()<=.01+1e-12),
                 'structural_keep_only':bool(structural_keep),'actual_keep_only_in_scope':bool(I==0)}
            harm_rows.append(row)
        fold_policy_rows.append({**job,'harm_penalty':main_settings['harm_penalty'],'threshold':main_settings['threshold'],
                                 'structural_keep_only':structural_keep,'clean_f1_loss_pp':result['clean_f1_loss_pp']})
        print(f'  Reproduced {dataset} {job["fold"]} seed {job["seed"]}: {N} windows.',flush=True)
    for dataset,expected_participants in [('dsads',8),('har',9)]:
        anchor=None
        for seed in SEEDS:
            parts=[frame for (ds,fold,s),frame in metadata.items() if ds==dataset and s==seed]
            assert len(parts)==(4 if dataset=='dsads' else 1)
            people=[set(part.subjects.tolist()) for part in parts]
            for i in range(len(people)):
                for j in range(i+1,len(people)):
                    assert not people[i]&people[j],'A participant is tested in multiple folds within a seed.'
            combined=pd.concat(parts).sort_values('window_ids').reset_index(drop=True)
            assert not combined.window_ids.duplicated().any()
            assert combined.subjects.nunique()==expected_participants
            if anchor is None:
                anchor=combined
            else:
                pd.testing.assert_frame_equal(combined,anchor,check_exact=True)
    frames=[pd.DataFrame(values) for values in [rows,summary,participants,harm_rows,fold_policy_rows]]
    assert len(frames[0])==2340 and len(frames[1])==540 and len(frames[3])==45
    for frame,name,keys,cols in [
        (frames[0],'per_condition',['dataset','fold','seed','method','condition'],NUMERIC),
        (frames[1],'test_summary',['dataset','fold','seed','method','scope'],NUMERIC),
        (frames[2],'per_participant',['dataset','fold','seed','subject','method','scope'],
         ['macro_f1_present_classes','accuracy','intervention_rate','helpful_rate','harmful_rate'])]:
        assert_table_parity(frame,pd.read_csv(root_file('notebook05_'+name+'.csv')),keys,cols)
    return frames


def aggregate_original(summary):
    seed=summary.groupby(['dataset','seed','method','scope'],sort=False)[NUMERIC].mean().reset_index()
    old_seed=pd.read_csv(root_file('notebook05_per_seed_summary.csv'))
    assert old_seed.complete_for_seed.all()
    assert_table_parity(seed,old_seed,['dataset','seed','method','scope'],NUMERIC)
    rows=[]
    for keys,part in seed.groupby(['dataset','method','scope'],sort=False):
        assert len(part)==3
        row=dict(zip(['dataset','method','scope'],keys))
        for col in NUMERIC:
            row[col+'_mean']=float(part[col].mean())
            row[col+'_seed_sd']=float(part[col].std(ddof=1))
        rows.append(row)
    aggregate=pd.DataFrame(rows)
    assert_table_parity(aggregate,pd.read_csv(root_file('notebook05_aggregate_summary.csv')),
                       ['dataset','method','scope'],[col+suffix for col in NUMERIC for suffix in ['_mean','_seed_sd']])
    return seed,aggregate


def paired_bootstrap(differences,indices):
    # Each entry is one original participant's difference, already averaged over
    # that participant's three seeds and all conditions of the requested scope.
    values=np.asarray(differences,dtype=np.float64)
    assert values.ndim==1 and indices.ndim==2 and indices.shape[1]==len(values)
    samples=values[indices].mean(axis=1)
    tail=(1-CONFIDENCE_LEVEL)/2
    low,high=np.quantile(samples,[tail,1-tail])
    return float(values.mean()),float(low),float(high),samples


def comparisons_and_uncertainty(summary,participant_rows,seed,aggregate):
    counts=participant_rows.groupby(['dataset','subject','method','scope']).seed.nunique()
    assert np.all(counts==3)
    subject=participant_rows.groupby(['dataset','subject','method','scope'],sort=False)[
        ['macro_f1_present_classes','accuracy','intervention_rate','helpful_rate','harmful_rate']].mean().reset_index()
    lookup=aggregate.set_index(['dataset','method','scope'])
    comparison_rows,run_differences,deletions=[],[],[]
    for dataset in ['dsads','har']:
        subjects=np.sort(subject[subject.dataset==dataset].subject.unique())
        rng=np.random.default_rng(BOOTSTRAP_SEED+zlib.crc32(dataset.encode()))
        indices=rng.integers(0,len(subjects),size=(BOOTSTRAP_RESAMPLES,len(subjects)),dtype=np.int16)
        distributions=[];labels=[]
        for scope in SCOPES:
            ss=subject[(subject.dataset==dataset)&(subject.scope==scope)].pivot(index='subject',columns='method',values='macro_f1_present_classes').loc[subjects]
            original=summary[(summary.dataset==dataset)&(summary.scope==scope)].pivot(index=['fold','seed'],columns='method',values='macro_f1')
            seed_scores=seed[(seed.dataset==dataset)&(seed.scope==scope)].pivot(index='seed',columns='method',values='macro_f1').loc[SEEDS]
            for comparator in COMPARATORS:
                delta=(ss.harm_aware_policy-ss[comparator]).to_numpy()*100
                point,lo,hi,samples=paired_bootstrap(delta,indices)
                a,b=lookup.loc[(dataset,'harm_aware_policy',scope)],lookup.loc[(dataset,comparator,scope)]
                run_delta=100*(original.harm_aware_policy-original[comparator])
                seed_delta=100*(seed_scores.harm_aware_policy-seed_scores[comparator])
                primary_delta=float(100*(a.macro_f1_mean-b.macro_f1_mean))
                assert abs(primary_delta-float(seed_delta.mean()))<1e-10
                for (fold,run_seed),value in run_delta.items():
                    run_differences.append({'dataset':dataset,'fold':fold,'seed':int(run_seed),'scope':scope,
                                            'comparator':comparator,'delta_macro_f1_pp':float(value)})
                delete_values=(delta.sum()-delta)/(len(delta)-1)
                for participant,value in zip(subjects,delete_values):
                    deletions.append({'dataset':dataset,'scope':scope,'comparator':comparator,
                        'removed_subject':int(participant),'remaining_participants':len(subjects)-1,
                        'participant_balanced_delta_pp':float(value),'refitting_performed':False})
                comparison_rows.append({'dataset':dataset,'scope':scope,'comparator':comparator,
                    'frozen_primary_comparator':comparator==PRIMARY[dataset],
                    'primary_policy_macro_f1_pct':float(a.macro_f1_mean*100),
                    'primary_comparator_macro_f1_pct':float(b.macro_f1_mean*100),
                    'primary_delta_macro_f1_pp':primary_delta,'paired_seed_delta_sd_pp':float(seed_delta.std(ddof=1)),
                    'positive_runs':int((run_delta>1e-10).sum()),'negative_runs':int((run_delta<-1e-10).sum()),
                    'tied_runs':int((run_delta.abs()<=1e-10).sum()),'total_runs':len(run_delta),
                    'participants':len(subjects),'participants_positive':int((delta>1e-10).sum()),
                    'participants_negative':int((delta<-1e-10).sum()),'participants_tied':int((np.abs(delta)<=1e-10).sum()),
                    'participant_balanced_delta_pp':point,'participant_bootstrap_ci_low_pp':lo,
                    'participant_bootstrap_ci_high_pp':hi,
                    'participant_deletion_min_delta_pp':float(delete_values.min()),
                    'participant_deletion_max_delta_pp':float(delete_values.max()),
                    'policy_harm_all_windows_pct':float(a.harmful_rate_mean*100),
                    'comparator_harm_all_windows_pct':float(b.harmful_rate_mean*100),
                    'primary_accuracy_delta_pp':float(100*(a.accuracy_mean-b.accuracy_mean)),
                    'ci_metric':'secondary participant-balanced Macro-F1 difference; not primary fold-mean Macro-F1'})
                labels.append(scope+'|'+comparator);distributions.append(samples)
        save_npz(OUTPUT/f'{dataset}_paired_participant_bootstrap.npz',subjects=subjects,participant_indices=indices,
                 comparison_labels=np.asarray(labels),delta_samples_pp=np.stack(distributions),
                 analysis_signature=np.asarray(ANALYSIS_SIGNATURE))
    return pd.DataFrame(comparison_rows),pd.DataFrame(run_differences),subject,pd.DataFrame(deletions)


def make_figures(aggregate,comparisons):
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.spines.top':False,
        'axes.spines.right':False,'pdf.fonttype':42,'ps.fonttype':42})
    methods=['keep','residual_gate_mask','confidence_select','harm_aware_policy']
    labels=['Keep','Residual mask','Confidence','Policy']
    colors=['#64748b','#2672a8','#df852b','#16846c']
    lookup=aggregate.set_index(['dataset','method','scope'])
    fig,axes=plt.subplots(2,3,figsize=(7.1,4.5),sharey=True)
    minimum=float((aggregate[aggregate.method.isin(methods)].macro_f1_mean-
                   aggregate[aggregate.method.isin(methods)].macro_f1_seed_sd).min()*100)
    maximum=float((aggregate[aggregate.method.isin(methods)].macro_f1_mean+
                   aggregate[aggregate.method.isin(methods)].macro_f1_seed_sd).max()*100)
    low,high=np.floor(minimum-2),np.ceil(maximum+2)
    for row,dataset in enumerate(['dsads','har']):
        for col,(scope,title) in enumerate([('clean','Clean'),('known_faults','Known faults'),('unseen_stuck','Unseen stuck')]):
            ax=axes[row,col]
            for x,(method,color) in enumerate(zip(methods,colors)):
                value=lookup.loc[(dataset,method,scope)]
                mean,sd=value.macro_f1_mean*100,value.macro_f1_seed_sd*100
                ax.errorbar(x,mean,yerr=sd,fmt='o',color=color,capsize=3,markersize=4)
                ax.annotate(f'{mean:.2f}',(x,mean+sd),xytext=(0,5),textcoords='offset points',ha='center',fontsize=6.5)
            ax.set_xticks(range(4),['K','RM','CS','HA'])
            ax.set_xlim(-.5,3.5);ax.set_ylim(low,high);ax.grid(axis='y',alpha=.18)
            ax.set_title(dataset.upper()+' — '+title,fontsize=8)
            if col==0:ax.set_ylabel('Macro-F1 (%)')
    handles=[Line2D([0],[0],marker='o',color=color,linestyle='',label=label) for color,label in zip(colors,labels)]
    fig.legend(handles=handles,loc='upper center',ncol=4,frameon=False,bbox_to_anchor=(.5,1.0),fontsize=8)
    fig.text(.5,.012,'Primary metric; error bars show SD across three saved seeds, not confidence intervals.',ha='center',fontsize=7)
    fig.tight_layout(rect=[0,.04,1,.92])
    for extension in ['pdf','png']:fig.savefig(OUTPUT/f'primary_macro_f1.{extension}',dpi=300,bbox_inches='tight')
    plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(7.1,4.1))
    scope_colors={'known_faults':'#2672a8','unseen_stuck':'#9c477c'}
    short={'keep':'Keep','residual_gate_mask':'Residual mask','confidence_select':'Confidence'}
    order=[(scope,base) for scope in ['known_faults','unseen_stuck'] for base in DISPLAY_COMPARATORS]
    for ax,dataset in zip(axes,['dsads','har']):
        part=comparisons[comparisons.dataset==dataset].set_index(['scope','comparator'])
        ticklabels=[]
        for y,(scope,base) in enumerate(order):
            r=part.loc[(scope,base)]
            color=scope_colors[scope]
            ax.plot([r.participant_bootstrap_ci_low_pp,r.participant_bootstrap_ci_high_pp],[y,y],color=color,lw=1.5)
            ax.scatter([r.participant_balanced_delta_pp],[y],color=color,s=22,zorder=3)
            ticklabels.append(('Known' if scope=='known_faults' else 'Stuck')+' vs '+short[base])
        ax.set_yticks(range(len(order)),ticklabels,fontsize=7);ax.invert_yaxis()
        ax.axvline(0,color='#6b7280',ls='--',lw=.8);ax.grid(axis='x',alpha=.18)
        ax.set_xlabel('Participant-balanced Macro-F1 difference (pp)',fontsize=7)
        ax.set_title(dataset.upper()+f' ({8 if dataset=="dsads" else 9} participants)',fontsize=8)
    fig.suptitle('Secondary participant-level effects with paired 95% bootstrap intervals',fontsize=9)
    fig.text(.5,.012,'Conditional on saved fits; exploratory intervals. These are not intervals for the primary metric.',ha='center',fontsize=7)
    fig.tight_layout(rect=[0,.04,1,.94])
    for extension in ['pdf','png']:fig.savefig(OUTPUT/f'participant_paired_intervals.{extension}',dpi=300,bbox_inches='tight')
    plt.close(fig)


def main():
    started=time.perf_counter()
    condition_rows,summary,participants,harm,policies=load_and_reproduce()
    seed,aggregate=aggregate_original(summary)
    comparisons,run_differences,subject_scores,deletions=comparisons_and_uncertainty(summary,participants,seed,aggregate)
    primary_runs=run_differences[[base==PRIMARY[ds] for ds,base in zip(run_differences.dataset,run_differences.comparator)]]
    assert_table_parity(primary_runs,pd.read_csv(root_file('notebook05_primary_comparison.csv')),
                       ['dataset','fold','seed','scope'],['delta_macro_f1_pp'])
    save_csv(OUTPUT/'notebook06_comparison_summary.csv',comparisons)
    save_csv(OUTPUT/'notebook06_harm_audit.csv',harm)
    save_csv(OUTPUT/'notebook06_run_differences.csv',run_differences)
    save_csv(OUTPUT/'notebook06_participant_scores.csv',subject_scores)
    save_csv(OUTPUT/'notebook06_participant_deletion.csv',deletions)
    save_csv(OUTPUT/'notebook06_selected_policies.csv',policies)
    severity=condition_rows.groupby(['dataset','seed','method','condition'],sort=False)[NUMERIC].mean().reset_index()
    severity=severity.groupby(['dataset','method','condition'],sort=False)[NUMERIC].mean().reset_index()
    save_csv(OUTPUT/'notebook06_per_condition_mean.csv',severity)
    make_figures(aggregate,comparisons)
    findings={}
    for dataset in ['dsads','har']:
        d=comparisons[comparisons.dataset==dataset]
        ha=harm[harm.dataset==dataset]
        pa=policies[policies.dataset==dataset]
        scope_findings={}
        for scope in SCOPES:
            hs=ha[ha.scope==scope]
            cs=d[d.scope==scope]
            scope_findings[scope]={
                'primary_policy_macro_f1_pct':float(cs.primary_policy_macro_f1_pct.iloc[0]),
                'primary_delta_vs_keep_pp':float(cs[cs.comparator=='keep'].primary_delta_macro_f1_pp.iloc[0]),
                'primary_delta_vs_confidence_pp':float(cs[cs.comparator=='confidence_select'].primary_delta_macro_f1_pp.iloc[0]),
                'primary_delta_vs_residual_mask_pp':float(cs[cs.comparator=='residual_gate_mask'].primary_delta_macro_f1_pp.iloc[0]),
                'runs_exceeding_1pct_harm':int((~hs.empirical_harm_le_1pct).sum()),'runs':len(hs),
                'maximum_run_harm_pct':float(hs.harm_all_windows.max()*100)}
        findings[dataset]={'participants':int(subject_scores[subject_scores.dataset==dataset].subject.nunique()),
            'structural_keep_only_configurations':int(pa.structural_keep_only.sum()),'configurations':len(pa),
            'scopes':scope_findings}
    conclusion={
        'full_study_reproduced_from_saved_predictions':True,'new_models_fit':0,'new_test_predictions_computed':0,
        'threshold_changes':0,'primary_metric_preserved':True,'original_seed_sd_is_not_ci':True,
        'secondary_bootstrap_intervals_are_not_primary_metric_ci':True,
        'empirical_1pct_harm_target_is_guaranteed':False,'significance_noninferiority_or_population_guarantee_claimed':False,
        'har_all_saved_main_choices_equal_keep':bool(np.all(harm[harm.dataset=='har'].actual_keep_only_in_scope)),
        'inference_compute_savings_established':False,'novelty_established_by_this_analysis':False}
    output_report={'code_version':CODE_VERSION,'analysis_signature':ANALYSIS_SIGNATURE,
        'source_study_signature':report['study_signature'],'configurations_reproduced':15,
        'condition_method_rows_reproduced':len(condition_rows),'summary_rows_reproduced':len(summary),
        'bootstrap_resamples':BOOTSTRAP_RESAMPLES,'bootstrap_seed':BOOTSTRAP_SEED,
        'primary_metric':plan['primary_metric'],'secondary_metric':plan['secondary_metric'],
        'bootstrap_method':plan['bootstrap'],'bootstrap_limitations':plan['bootstrap_limitations'],
        'findings':findings,'claim_audit':conclusion,'analysis_seconds':time.perf_counter()-started,
        'figures':['primary_macro_f1.pdf','primary_macro_f1.png','participant_paired_intervals.pdf','participant_paired_intervals.png'],
        'output_directory':str(OUTPUT)}
    save_json(OUTPUT/'notebook06_run_report.json',output_report)
    artifacts=[{'filename':str(p.relative_to(OUTPUT)),'bytes':p.stat().st_size,'sha256':file_sha256(p)}
        for p in sorted(OUTPUT.rglob('*')) if p.is_file() and not p.name.endswith('.partial')
        and p.name!='notebook06_artifact_inventory.json']
    save_json(OUTPUT/'notebook06_artifact_inventory.json',artifacts)
    show=comparisons[comparisons.comparator.isin(DISPLAY_COMPARATORS)&comparisons.scope.isin(['known_faults','unseen_stuck'])]
    print('\nNOTEBOOK 06 COMPLETE',flush=True)
    print('All 15 configurations and all Notebook 05 summary metrics reproduced.',flush=True)
    print('PRIMARY EFFECTS | original fold/seed-averaged Macro-F1 | percentage points',flush=True)
    print(show[['dataset','scope','comparator','primary_policy_macro_f1_pct','primary_comparator_macro_f1_pct',
                'primary_delta_macro_f1_pp','positive_runs','negative_runs','tied_runs']]
          .to_string(index=False,float_format=lambda x:f'{x:.3f}'),flush=True)
    print('\nSECONDARY EFFECTS | participant-balanced Macro-F1 | conditional 95% participant bootstrap intervals',flush=True)
    print(show[['dataset','scope','comparator','participants','participant_balanced_delta_pp',
                'participant_bootstrap_ci_low_pp','participant_bootstrap_ci_high_pp']]
          .to_string(index=False,float_format=lambda x:f'{x:.3f}'),flush=True)
    print('The secondary intervals do not apply to the primary effects above. No significance or population guarantee is claimed.',flush=True)
    for dataset,data in findings.items():
        print(dataset.upper(),json.dumps(data),flush=True)
    print('New training: 0 | New test inference: 0 | Threshold changes: 0 | GPU required: No',flush=True)
    print('Outputs:',OUTPUT,flush=True)
    print('Share this final output and notebook06_comparison_summary.csv, notebook06_run_report.json, notebook06_harm_audit.csv.',flush=True)


main()


Mounted at /content/drive
NOTEBOOK 06 | saved-prediction audit and participant uncertainty | CPU | one executable cell
No training, new test predictions, threshold changes, or GPU required.
Analysis plan recorded: 4d786da16740b861
  Reproduced dsads fold_0 seed 2026: 2280 windows.
  Reproduced dsads fold_1 seed 2026: 2280 windows.
  Reproduced dsads fold_2 seed 2026: 2280 windows.
  Reproduced dsads fold_3 seed 2026: 2280 windows.
  Reproduced har fold_0 seed 2026: 2947 windows.
  Reproduced dsads fold_0 seed 2027: 2280 windows.
  Reproduced dsads fold_1 seed 2027: 2280 windows.
  Reproduced dsads fold_2 seed 2027: 2280 windows.
  Reproduced dsads fold_3 seed 2027: 2280 windows.
  Reproduced har fold_0 seed 2027: 2947 windows.
  Reproduced dsads fold_0 seed 2028: 2280 windows.
  Reproduced dsads fold_1 seed 2028: 2280 windows.
  Reproduced dsads fold_2 seed 2028: 2280 windows.
  Reproduced dsads fold_3 seed 2028: 2280 windows.
  Reproduced har fold_0 seed 2028: 2947 windows.

NOTEBOOK 